<div style="background: #0f172a; padding: 40px 32px; border-radius: 12px; margin-bottom: 24px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">

<p style="color: #94a3b8; font-size: 12px; letter-spacing: 0.18em; text-transform: uppercase; font-weight: 500; margin: 0 0 20px 0;">
  FINANCIAL CONTEXT &middot; GLOSSARY &middot; NEW TO FINANCE?
</p>

<h1 style="font-family: Georgia, 'Times New Roman', serif; font-size: 44px; line-height: 1.1; color: #f8fafc; font-weight: 400; margin: 0 0 8px 0;">
  Read the <em style="color: #5eead4; font-style: italic;">Glossary</em> first
</h1>

<p style="color: #cbd5e1; font-size: 16px; line-height: 1.6; margin: 24px 0 0 0; max-width: 720px;">
  This notebook assumes familiarity with concepts like <span style="color: #f8fafc; font-weight: 500;">10-K filings</span>, <span style="color: #f8fafc; font-weight: 500;">GICS sectors</span>, <span style="color: #f8fafc; font-weight: 500;">evidence passages</span>, and retrieval metrics (Recall@k, MRR, NDCG, MAP). If any of these terms feel unfamiliar, read the glossary first to get the most out of the analysis below.
</p>

<p style="margin: 28px 0 0 0;">
  <a href="../docs/CONTEXT.md" style="color: #5eead4; font-size: 15px; text-decoration: none; border-bottom: 1px solid #5eead4; padding-bottom: 2px;">
    &rarr;&nbsp;&nbsp;Open: docs/CONTEXT.md &mdash; Financial context and glossary
  </a>
</p>

</div>

---

# FinanceBench — Dataset exploration

**Stage 1 — Baselines · Block: FinanceBench Loader**

Exploratory notebook for the [`PatronusAI/financebench`](https://huggingface.co/datasets/PatronusAI/financebench) dataset: 150 QA pairs grounded in real SEC filings (10-K, 10-Q, 8-K) and earnings releases from publicly traded companies.

**Notebook goal**: understand the dataset's structure, coverage, and difficulty before parsing PDFs and building the indexable corpus used by downstream retrieval baselines.

---

## Setup

Load the dataset from HuggingFace. On the first run it downloads the Parquet files (~a few MB) and caches them in `~/.cache/huggingface/datasets/`. Subsequent runs are instant.

In [ ]:
# load_dataset is HuggingFace's magic function to download datasets from the Hub.
# Takes the identifier "<org>/<dataset-name>", same convention as models.
from datasets import load_dataset

# load_dataset() returns a DatasetDict — a dict-like object where:
#   - keys   = dataset splits (typically "train", "validation", "test")
#   - values = Dataset objects (HF's own class, optimized with Apache Arrow)
# The first call downloads the Parquet files and caches them in ~/.cache/huggingface/datasets/.
# Subsequent calls are instant (read from local cache).
ds = load_dataset("PatronusAI/financebench")

# FinanceBench is eval-only and ships with only 1 split. By library convention,
# any single split is named "train" even if its actual use is evaluation.
# IMPORTANT: we will NOT create train/test splits over FinanceBench — it stays
# intact as our sacred eval set. For fine-tuning we use a DIFFERENT 10-K source (Stage 3).
train = ds["train"]

# Inspect the basic structure:
#   type(ds).__name__   → "DatasetDict" (the container)
#   list(ds.keys())     → available splits
#   train.num_rows      → number of QA pairs (we expect 150)
#   train.column_names  → 15 fields (financebench_id, question, answer, evidence, etc.)
print(f"Type:    {type(ds).__name__}")
print(f"Splits:  {list(ds.keys())}")
print(f"Rows:    {train.num_rows}")
print(f"Columns: {len(train.column_names)}")

---

## Section 2 — Structure of the 150 QA pairs

We explore the dataset across **four dimensions** to understand whether it's a robust evaluation set:

1. **Documentary diversity** — how many companies, sectors, and years does it cover?
2. **Question types** — what reasoning is required (extraction vs. calculation)?
3. **Lengths** — how long are questions, answers, and evidence passages?
4. **Evidence distribution** — how many supporting passages per question?

The goal is to be able to defend the dataset in a technical interview ("is FinanceBench a robust eval set?") with data, not just intuition.

### 2.0 Dataset schema

The **15 fields** that make up each record in the dataset, in the order defined by the official schema. Useful as a quick reference when working on later sub-sections (lengths, evidence distribution, etc.).

In [ ]:
# train.column_names returns the list of columns in the order defined by the schema.
# enumerate(..., 1) starts the counter at 1 (instead of 0) so the list reads as a
# human-friendly index (1, 2, 3...) instead of a programmer index (0, 1, 2...).
# Format spec ":2d" → integer right-aligned to 2 characters (handles 1-99 without misalignment).
for i, col in enumerate(train.column_names, 1):
    print(f"{i:2d}. {col}")

#### Sample row — minimal QA structure

We print the **first record** of the dataset with all its fields. This is the **atomic unit** we'll be working with: our RAG system consumes `question`, retrieves chunks, and compares them against `evidence`.

For long fields (`question`, `evidence_text`, `justification`) we truncate the content and report the real length in parentheses, so you get a feel for the size without flooding the output.

In [ ]:
# Take the FIRST record of the dataset as a representative example.
# train[0] returns a dict {column_name: value} — the atomic unit of the dataset.
row = train[0]

# Walk through each field of the record to display it in a readable format.
# Display logic:
#   - Short strings (≤ 250 chars) → printed in full
#   - Long strings                → truncated with ellipsis AND real length reported
#   - Lists (the `evidence` case) → unpacked into their inner dicts
#   - Other types (int, None)     → printed as-is
for key, value in row.items():
    if isinstance(value, str) and len(value) > 250:
        # Report the real length in parentheses to give a sense of the size.
        print(f"{key}: ({len(value)} chars)")
        print(f"  {value[:250]}...")
    elif isinstance(value, list):
        # `evidence` is a list[dict] — show each item indexed.
        print(f"{key}: ({len(value)} items)")
        for i, item in enumerate(value):
            for k, v in item.items():
                if isinstance(v, str) and len(v) > 250:
                    print(f"  [{i}] {k}: ({len(v)} chars)")
                    print(f"      {v[:250]}...")
                else:
                    print(f"  [{i}] {k}: {v}")
    else:
        # Short fields: int (doc_period), None (domain_question_num), short strings.
        print(f"{key}: {value}")

### 2.1 Documentary diversity

**Core question**: does the dataset cover enough companies, sectors, and years for our metrics to generalize, or is it biased toward a niche?

If all 150 QA pairs were about a single company or a single sector (e.g., Tech only), our embedders could "memorize" that niche's vocabulary and produce inflated metrics that don't reflect real performance on other financial domains.

**Metrics we will measure**:

- # of distinct companies (more is better diversity)
- # of GICS sectors covered (out of 11 possible)
- Temporal range (fiscal years covered)
- Mix of document types (10-K, 10-Q, 8-K, earnings releases)

In [ ]:
# Counter (from collections) counts occurrences of each value in a list/iterable.
# Result: dict-like {value: count}, with handy methods like .most_common(n).
from collections import Counter

# train["company"] returns the entire column as a list (Arrow → Python list).
# This does NOT load the whole dataset into RAM because we extract only one column.
# Counter() turns the list into a mapping {company: # appearances}.
companies = Counter(train["company"])
sectors = Counter(train["gics_sector"])
years = Counter(train["doc_period"])
doc_types = Counter(train["doc_type"])

# === Block 1: Overview ===
# len(companies) → how many DISTINCT companies (not the total row count).
# min/max over years works because the keys are ints (fiscal years).
# dict(doc_types) converts the Counter to a regular dict for cleaner printing.
print("=" * 60)
print("DOCUMENTARY DIVERSITY — overview")
print("=" * 60)
print(f"Distinct companies: {len(companies)}")
print(f"GICS sectors:       {len(sectors)} (out of 11 possible)")
print(f"Years covered:      {min(years)}–{max(years)} ({len(years)} years)")
print(f"Document types:     {dict(doc_types)}")
print()

# === Block 2: Top 10 companies ===
# .most_common(10) returns the 10 most frequent values sorted descending.
# Format: [(value, count), ...] — we unpack in the for loop.
# This shows whether any single company dominates (which would bias the eval).
print("=" * 60)
print("TOP 10 COMPANIES BY # OF QUESTIONS")
print("=" * 60)
for company, count in companies.most_common(10):
    # f-string format spec: ":30s" = string padded to 30 chars, ":3d" = int in 3 chars.
    print(f"  {company:30s} {count:3d}")
print()

# === Block 3: Distribution by sector with ASCII bar chart ===
# .most_common() with no argument returns ALL items sorted by frequency desc.
# We compute % and draw a bar with "█" characters (1 char = 2%).
print("=" * 60)
print("DISTRIBUTION BY GICS SECTOR")
print("=" * 60)
for sector, count in sectors.most_common():
    pct = count / 150 * 100
    bar = "█" * int(pct / 2)  # each block = 2% (50% → 25 blocks max)
    print(f"  {sector:30s} {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 4: Temporal distribution ===
# Iterate over years in ASCENDING order to see chronological evolution.
# sorted(years.keys()) guarantees ascending order.
# We expect a strong concentration in 2022-2023 (when Patronus built the benchmark).
print("=" * 60)
print("DISTRIBUTION BY FISCAL YEAR")
print("=" * 60)
for year in sorted(years.keys()):
    count = years[year]
    pct = count / 150 * 100
    bar = "█" * int(pct / 2)
    print(f"  {year}  {count:3d}  {bar} {pct:.1f}%")

#### Findings

**✅ Solid coverage:**

- **32 distinct companies**, no dominant outlier (max: PepsiCo with 11/150 = 7.3%) — good diversity, so our metrics aren't capturing the idiosyncrasies of a single firm.
- **9 of 11 GICS sectors** represented — covers most of the economy (only Energy and Real Estate are absent). If our embedders generalize here, they should generalize to almost any corporate financial domain.
- **10 years covered (2015–2024)** — wide temporal window.

**⚠️ Things to document as limitations:**

**1. The dataset is NOT only 10-K**, it contains a mix:

| Type | # | % | Nature |
|---|---:|---:|---|
| `10k` | 112 | 74.7% | Annual reports (audited, formal) |
| `10q` | 15 | 10.0% | Quarterly reports (unaudited) |
| `Earnings` | 14 | 9.3% | **Not SEC filings** — voluntary investor communications |
| `8k` | 9 | 6.0% | Event-driven reports |

**Technical decision (locked)**: keep the **150 mixed records** without filtering. Reason: FinanceBench is a public, standard evaluation set used in the literature; modifying it would make our results incomparable to other papers. Source heterogeneity is documented as a feature of the dataset, not as a bug.

**2. Temporal bias toward 2022–2023**: 64% of the questions target filings from those two years (2022: 40.7%, 2023: 23.3%). This reflects when Patronus AI built the benchmark. We **call this out in the final README** as a limitation: the system is being evaluated on recently published documentation.

> 📝 These decisions will be formally recorded in `docs/chunking_decisions.md` once Sub-block 6 of the current block is closed.

### 2.2 Question types

**Core question**: what kind of reasoning does each query require? Is it simple extraction ("what was the revenue?") or complex calculation ("how much did the margin grow YoY?")? This determines how challenging the dataset is.

Patronus AI labeled each question across **two independent fields** (not difficulty hierarchies — they are orthogonal dimensions):

- `question_type` → **origin / style** of the question
- `question_reasoning` → **type of reasoning** required to answer it

**Why did Patronus pick these categories?** Because the PDF of a 10-K has **3 distinct types of content**, and each category reflects one. The classification is NOT arbitrary — it follows the structure of the actual document.

---

#### `question_type` — origin / style of the question

##### 🟦 `metrics-generated`

**What in the PDF generates them**: the **standardized financial statements** of the 10-K (Income Statement, Balance Sheet, Cash Flow Statement) are tables with **identical structure across all companies** (SEC regulation). They have lines with standard names: `Net sales`, `Cost of goods sold`, `Capital expenditures`, `Long-term debt`, etc.

**How they're built systematically**: Patronus took a list of standard metrics and applied **rigid templates** of the form:

> *"What is the {metric} for {company} in fiscal year {year}? Answer in {unit}."*

A single template generates 50+ questions by varying the parameters (3M in 2018, AMD in 2022, Pfizer in 2023, etc.).

**Real example from 3M's PDF (2018 10-K, page 59)**:
- In the **Statement of Cash Flows** there's a line: `Purchases of property, plant and equipment ............ $(1,577)`
- Generated question: *"What is the FY2018 capital expenditure amount in USD millions for 3M?"*
- Answer literal in the PDF: **$1,577M**

**Why Patronus created it**: it guarantees **systematic coverage** of the metrics most used in financial analysis, and lets you **compare results across companies** with the same question varying only the subject.

##### 🟨 `domain-relevant`

**What in the PDF generates them**: beyond the tables, the 10-K has **mandatory narrative sections** where the company describes in free prose its operations, risks, and strategy. The main ones are:

- **Item 1 (Business)** — business description
- **Item 1A (Risk Factors)** — list of material risks
- **Item 7 (MD&A)** — management's discussion and analysis

These sections are **predictable in their existence** (every 10-K has them) but **variable in their content** (each company describes its own risks).

**How they're built**: humans wrote questions with **clear focus on these sections** but without sticking to a template. The questions are thematically predictable ("what risks does it identify?", "what is the growth strategy?") but the format is free.

**Real example from 3M's PDF (2018 10-K, Item 1A)**:
- The PDF has several paragraphs describing risks: PFAS contamination, pending litigation, supply chain disruption, etc.
- Question: *"What are the main risks 3M identifies in its 2018 annual report?"*
- Answer: **requires READING and SYNTHESIZING several prose paragraphs** — there's no single value to extract.

**Why Patronus created it**: questions about financial statements are easy to template, but real analysts also ask about **the qualitative context** (strategy, risks, governance). This category covers that ground.

##### 🟥 `novel-generated`

**What in the PDF generates them**: information that **crosses sections** of the 10-K in ways an analyst judges relevant but that don't follow an established pattern. For example, combining:

- A metric from the Income Statement
- A narrative from MD&A
- A risk from the Risk Factors section

**How they're built**: humans wrote **completely open** questions, simulating how a junior analyst asks a colleague or how a curious investor explores a 10-K without a predefined agenda.

**Real example from Pfizer's PDF (2022 10-K)**:
- The PDF has: (a) R&D spend in the Income Statement, (b) description of the oncology pipeline in Item 1 Business, (c) competitive risks in Item 1A.
- Question: *"How is Pfizer's R&D pipeline positioning the company in oncology?"*
- Answer: **requires crossing 3 sections of the PDF** — the financial tables (how much is invested), the Business narrative (what is being built), and the Risk Factors (what competitive threats exist).

**Why Patronus created it**: it's the **stress test** of the benchmark. If your RAG system does well on `metrics-generated` (templated, predictable questions) but **collapses on `novel-generated`** (free-form questions), that reveals the system **memorizes the prompt format** instead of understanding semantics.

---

#### `question_reasoning` — required reasoning

**What does this dimension classify?** How much cognitive work the system has to do **after** finding the correct chunk. The 3 categories reflect **where the answer lives in the PDF**: literal in the text, computed from PDF data, or derived by combining multiple pieces.

##### 🔵 `Information extraction`

**How it looks in the PDF**: the answer is **literal in a table cell or in a direct mention in prose**. The system just needs to find the correct chunk and return the text.

**Real example**:
- 3M's PDF (2018 10-K, Income Statement): line `Net sales ............... $32,765`
- Question: *"What was 3M's revenue in FY2018?"*
- Direct answer: **$32,765M** — already literal in the PDF, no calculation needed.

##### 🟢 `Numerical reasoning`

**How it looks in the PDF**: the answer is **NOT literal** in the PDF — the document contains the **inputs** (2 or more values), but the result has to be **computed**.

**Real example**:
- 3M's PDF: `Net sales 2017 = $31,657` and `Net sales 2018 = $32,765`
- Question: *"What was 3M's revenue YoY growth from 2017 to 2018?"*
- Answer: **NOT in the PDF** — must be computed: `(32,765 - 31,657) / 31,657 = 3.5%`.

The RAG system **must retrieve the chunk** with both numbers (ideally the full table showing the year-over-year comparison). The calculation is done by the downstream LLM.

##### 🟣 `Logical reasoning (multi-step)`

**How it looks in the PDF**: the answer requires **combining multiple pieces of information**, possibly from **different sections of the PDF** (e.g., Cash Flow Statement + Income Statement). You have to extract data, compute intermediate ratios, compare results, and emit a judgment.

**Real example**:
- Question: *"Did 3M's CapEx grow faster than its revenue between 2017 and 2018?"*
- Inputs in the PDF (4 numbers, 2 different sections):
  - **Income Statement**: Revenue 2017 = $31,657M, Revenue 2018 = $32,765M
  - **Cash Flow Statement**: CapEx 2017 = $1,373M, CapEx 2018 = $1,577M
- Process:
  1. Extract the 4 numbers
  2. Compute CapEx growth = `(1,577 - 1,373) / 1,373 = 14.9%`
  3. Compute Revenue growth = `(32,765 - 31,657) / 31,657 = 3.5%`
  4. Compare: 14.9% > 3.5% → **Yes, CapEx grew faster**

The RAG system **must retrieve chunks from 2 different sections** of the 10-K. This connects directly with what we'll see in Sub-block 2.4: **~23% of the dataset has 2-3 distinct evidences** because multi-step questions require information from various places in the PDF.

> 💡 **Summary table + glossary of categories**: see [`docs/CONTEXT.md` § How Patronus categorized the questions](../docs/CONTEXT.md#how-patronus-categorized-the-questions).

In [ ]:
# Counter over question_type — distribution of the 3 general categories.
q_types = Counter(train["question_type"])

# question_reasoning can be None in ~33% of the rows (all novel-generated).
# Replace None with the string "(None)" in a generator expression so Counter
# doesn't choke when iterating, and the output clearly shows unclassified rows.
q_reasoning = Counter(
    str(x) if x is not None else "(None)"
    for x in train["question_reasoning"]
)

# === Block 1: question_type ===
print("=" * 65)
print("QUESTION_TYPE — general question category")
print("=" * 65)
for qt, count in q_types.most_common():
    pct = count / 150 * 100
    bar = "█" * int(pct / 2)
    print(f"  {qt:30s} {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 2: question_reasoning ===
print("=" * 65)
print("QUESTION_REASONING — required reasoning type")
print("=" * 65)
for qr, count in q_reasoning.most_common():
    pct = count / 150 * 100
    bar = "█" * int(pct / 2)
    print(f"  {qr:35s} {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 3: Cross-tab question_type × question_reasoning ===
# defaultdict(lambda: defaultdict(int)) creates a NESTED dict that auto-initializes
# any missing key with 0. Useful for accumulating counts without existence checks.
from collections import defaultdict
cross = defaultdict(lambda: defaultdict(int))
for r in train:
    qr_key = r["question_reasoning"] if r["question_reasoning"] is not None else "(None)"
    cross[r["question_type"]][qr_key] += 1

print("=" * 65)
print("CROSS-TAB question_type × question_reasoning")
print("=" * 65)
for qt in sorted(cross.keys()):
    print(f"\n[{qt}]")
    # Sort by count desc within each category → most common reasoning shows first.
    for qr, c in sorted(cross[qt].items(), key=lambda x: -x[1]):
        print(f"  {qr:50s} {c:3d}")

#### Findings

**1. `question_type` is PERFECTLY balanced** (50/50/50):

| Category | # | Nature |
|---|---:|---|
| `metrics-generated` | 50 | Templated questions about metrics (CapEx, revenue, margin) |
| `domain-relevant` | 50 | Broader domain questions (strategy, risks, governance) |
| `novel-generated` | 50 | Novel questions, no template |

Patronus AI designed the dataset with equal quotas — good news for our metrics, no bias toward any single type.

**2. `question_reasoning` shows real heterogeneity**:

- ~29% **Numerical reasoning** (calculations)
- ~21% **Information extraction** (direct lookup)
- ~17% **Logical hybrids** (combinations of Logical + Numerical multi-step)
- 33% **`None`** (all `novel-generated`, unclassified)

**3. Cross-tab key insight**:

- `metrics-generated` → 72% Numerical + 28% Information extraction (templated questions are calculations or lookups)
- `domain-relevant` → broader diversity, with many multi-step hybrid cases (the most complex questions)
- `novel-generated` → all with `reasoning = None` (Patronus left them unclassified)

**Project implication**: the ~29% Numerical reasoning requires that the RAG system not only retrieves the correct chunk, but that the downstream LLM performs the calculation. Since Stage 1-2 measures **retrieval only**, we focus on whether the retrieved chunk contains the necessary numbers — the calculation is the generation layer's responsibility (out of scope here, addressed in Eslabón 2).

> 📝 Defensive against `None`: any code iterating `question_reasoning` must handle None values (33% of the dataset).

#### Tagging in action — seeing how it applies to real questions

So far we saw the **aggregate counts** (50/50/50, etc.) and the **hypothetical examples** from the PDF. Now we close the pedagogical loop: we take **3 real questions from the dataset** (one per `question_type`) and display **all their tags + metadata + an evidence fragment**.

This shows how Patronus applied the categorization to concrete questions, and how each category has a distinct "feel" when you read it:

- **`metrics-generated`** feels like a standardized exam (predictable structure, exact numerical value)
- **`domain-relevant`** feels like an analytical conversation (clear focus, free format)
- **`novel-generated`** feels like a natural question (no pattern, exploratory)

> 💡 **Tip**: change the `random.seed(42)` to another number in the cell below to explore other examples from the dataset and see the real diversity of each category.

In [ ]:
import random

# We fix the seed for reproducibility: every time you run this cell, the same
# 3 examples are chosen. Change the seed to explore other dataset cases.
random.seed(42)


def show_tagged_example(category: str, max_evidence_chars: int = 250) -> None:
    """Show ONE random example of the indicated question_type category,
    displaying all its tags + metadata + a fragment of the evidence.

    Educational: connects the theory (question type + reasoning) with the real dataset.
    """
    # Filter the dataset by question_type and pick one at random.
    filtered = [r for r in train if r["question_type"] == category]
    example = random.choice(filtered)

    print("=" * 80)
    print(f"📌 EXAMPLE: question_type = {category}")
    print("=" * 80)

    print(f"\n💬 Question:\n  {example['question']}\n")

    # The 2 main tags we've been analyzing above.
    print("🏷️  Tags:")
    print(f"  question_type:       {example['question_type']}")
    print(f"  question_reasoning:  {example['question_reasoning']}\n")

    # Source document metadata — useful to locate the evidence in the actual PDF.
    print("📄 Source document:")
    print(f"  company:             {example['company']}")
    print(f"  doc_name:            {example['doc_name']}")
    print(f"  doc_type / period:   {example['doc_type']} / {example['doc_period']}")
    print(f"  gics_sector:         {example['gics_sector']}\n")

    # Expected answer. For Information extraction it's a literal value;
    # for Numerical reasoning it's usually a computed number; for Logical multi-step
    # it can be a yes/no judgment or a textual conclusion.
    print(f"✅ Answer:\n  {example['answer']}\n")

    # Evidence: PDF passage(s) that justify the answer. Truncated for readability.
    # If there's more than 1 item, these are the multi-evidence QAs (~23% of dataset).
    print(f"📑 Evidence ({len(example['evidence'])} item(s)):")
    for i, item in enumerate(example["evidence"]):
        page = item.get("evidence_page_num", "N/A")
        text = item.get("evidence_text", "")
        truncated = text[:max_evidence_chars] + ("..." if len(text) > max_evidence_chars else "")
        print(f"  [{i}] page {page}:")
        # Indent the evidence text so it stands out visually.
        for line in truncated.split("\n")[:6]:
            print(f"      {line}")
    print()


# Call the function for the 3 question_type categories.
# Each call picks a different random example (reproducible thanks to the seed).
for cat in ["metrics-generated", "domain-relevant", "novel-generated"]:
    show_tagged_example(cat)
    print()

#### Important clarification about `question_reasoning = None`

A common intuition when looking at the data: *"`novel-generated` questions have `reasoning = None` because they require more complex post-processing from the LLM."*

**That's NOT accurate.** `question_reasoning = None` means **Patronus did not classify them** by reasoning type — NOT that they necessarily require more processing. A `novel-generated` question can be simple extraction or complex numerical; Patronus left them free-form because their diversity makes them hard to bucket.

**But the broader intuition IS correct**: ALL questions involving numerical or logical reasoning (~46% of the dataset) need post-processing from the downstream LLM. That's **not retrieval** — it's generation, living in a different phase of the RAG pipeline.

> 💡 **The 3 RAG pipeline phases (retrieval / reranking / generation) and what Stage 1-2 covers vs Eslabón 2**: see [`docs/CONTEXT.md` §10 — The RAG pipeline](../docs/CONTEXT.md#10-the-rag-pipeline--retrieval-reranking-generation).

### 2.3 Lengths (questions, answers, evidence)

**Core question**: how long are the dataset's texts? This defines two critical things for the RAG system:

1. **Embedder token budget** — modern embedders have a limit (typically 512 tokens). If our chunks or queries exceed it, we must truncate (losing information) or switch models.
2. **Appropriate chunking strategy** — if `evidence` passages are very long, we need bigger chunks OR retrieval that returns multiple chunks to cover the full evidence.

**What we measure**:

- Distribution (min, p25, median, p75, p95, max, mean) of each field in chars
- Token estimation (rule of thumb: 1 token ≈ 4 chars in English)
- Distribution of # of items per evidence (1, 2, or 3 passages per question)

We use `statistics.quantiles` from stdlib (no pandas/numpy) — the dataset is small, no need for heavy artillery.

> 💡 **Quick refresher**: if the terms `min`, `p25`, `median`, `p75`, `p95`, `max`, `mean` aren't familiar, read [`docs/CONTEXT.md` § 9 — Descriptive statistics primer](../docs/CONTEXT.md#9-descriptive-statistics-primer-min-mean-median-percentiles) before continuing.

In [ ]:
import statistics as stats

# Helper to print the distribution of a list of numeric values in a compact format.
# Reports: count, min, p25, median, p75, p95, max, mean.
#   stats.quantiles(values, n=4)  → quartiles: [p25, p50, p75]
#   stats.quantiles(values, n=20) → vigintiles: index 18 = p95 (19/20)
def describe(name, values):
    p25 = int(stats.quantiles(values, n=4)[0])
    p75 = int(stats.quantiles(values, n=4)[2])
    p95 = int(stats.quantiles(values, n=20)[18])
    print(f"{name:25s}  n={len(values):3d}  min={min(values):5d}  p25={p25:5d}  "
          f"median={int(stats.median(values)):5d}  p75={p75:5d}  p95={p95:5d}  "
          f"max={max(values):5d}  mean={int(stats.mean(values)):5d}")


# === Compute lengths in CHARS ===
# question, answer: simple strings, len() directly.
question_lens = [len(r["question"]) for r in train]
answer_lens = [len(r["answer"]) for r in train]

# justification can be None (~25% of dataset). We treat None as length 0.
# This keeps the count at 150 rows and honestly reflects missing justifications.
justification_lens = [len(r["justification"]) if r["justification"] else 0 for r in train]

# evidence is list[dict]. For each QA, we sum the chars of ALL evidence_text items.
# This reflects the "total evidence load" the system would need to retrieve.
evidence_total_lens = [
    sum(len(item["evidence_text"]) for item in r["evidence"])
    for r in train
]

# === Block 1: Distribution in chars ===
print("=" * 110)
print("LENGTHS (in chars) — distribution")
print("=" * 110)
describe("question", question_lens)
describe("answer", answer_lens)
describe("justification", justification_lens)
describe("evidence (sum per QA)", evidence_total_lens)
print()

# === Block 2: Token estimation ===
# Rule of thumb for English: ~4 chars per token on average.
# (Spanish: ~3 chars/token. Source code: ~2 chars/token.)
# For exact tokens, you'd pass the text through the target model's tokenizer.
print("=" * 110)
print("TOKEN estimation (rule of thumb: 1 token ≈ 4 chars in English)")
print("=" * 110)
print(f"  question      median ~{int(stats.median(question_lens)/4):3d} tokens   p95 ~{int(stats.quantiles(question_lens, n=20)[18]/4):3d} tokens")
print(f"  answer        median ~{int(stats.median(answer_lens)/4):3d} tokens   p95 ~{int(stats.quantiles(answer_lens, n=20)[18]/4):3d} tokens")
print(f"  evidence/QA   median ~{int(stats.median(evidence_total_lens)/4):3d} tokens   p95 ~{int(stats.quantiles(evidence_total_lens, n=20)[18]/4):3d} tokens")
print()

# === Block 3: # of evidence items per question ===
# How many evidence pieces each QA has. Schema says 1-3 items.
evidence_counts = [len(r["evidence"]) for r in train]
counts_dist = Counter(evidence_counts)

print("=" * 110)
print("# of evidence items per question")
print("=" * 110)
for n in sorted(counts_dist.keys()):
    c = counts_dist[n]
    pct = c / 150 * 100
    bar = "█" * int(pct / 2)
    print(f"  {n} items  {c:3d}  {bar} {pct:.1f}%")

#### Findings

**1. Questions and answers are small, evidence is the diva of the show**:

| Field | median | p95 | max | Estimated tokens (median) |
|---|---:|---:|---:|---|
| `question` | 137 chars | 356 | 592 | ~34 tokens |
| `answer` | 50 chars | 285 | 609 | ~12 tokens |
| `justification` | 100 chars | 488 | 703 | ~25 tokens |
| **`evidence` (sum per QA)** | **1,450 chars** | **4,194** | **12,123** | **~362 tokens** |

- Questions and answers fit comfortably in any embedder (max 512 tokens).
- `justification` may be empty for ~25% of the dataset (`min=0`, `p25=0`).
- `evidence` has a **very long tail**: the outlier reaches 12,123 chars (~3,030 tokens).

**2. Critical chunking implication**:

If we chunk at **512 tokens (~2,048 chars)** — the standard convention for BERT-family embedders:

| Evidence percentile | Fits in 1 chunk? |
|---|---|
| Median (1,450 chars / ~362 tokens) | ✅ Plenty of room |
| p75 (2,267 chars / ~566 tokens) | ⚠️ Just barely overflows → needs 2 chunks |
| p95 (4,194 chars / ~1,048 tokens) | ❌ Needs 2-3 chunks |
| Max (12,123 chars / ~3,030 tokens) | 🚨 Outlier — needs 6+ chunks |

For ~25-30% of the dataset, the full evidence does NOT fit in 1 chunk. **This validates our choice to measure Recall@k with k=5,10** (not just k=1) and to include **MAP** as a metric — both reward retrieving multiple chunks when the evidence spans across them.

**3. # of evidence items per question**:

- **76.7%** has **1 item** (1 PDF passage justifies the answer)
- **20.7%** has **2 items**
- **2.7%** has **3 items**

**~23% of the dataset has multiple distinct evidences**. These are the cases where MRR misleads (it only looks at the first hit) and **MAP is honest** (it looks at all hits).

> 📝 These measurements will inform `docs/chunking_decisions.md` (sub-block 6) when we justify the chosen chunk size.

### 2.4 Evidence distribution

**Core question**: when the RAG system performs retrieval, **how deep into the 10-K does it have to look?** Do answers live at the beginning (executive summary), in the middle (MD&A + financial statements), or scattered throughout?

Three dimensions we measure:

1. **Distribution of `evidence_page_num`** — which pages do evidences live on?
2. **Multi-evidence locality** — when a QA has 2-3 evidences, do they come from nearby or far-apart pages?
3. **Ratio `evidence_text` vs `evidence_text_full_page`** — is the evidence almost the whole page or a small fragment?

**Why it matters**:

- **If evidences live in a bounded range** (e.g., pages 0-100), there's no point indexing the whole document — we can truncate initial pages (cover, TOC) and final ones (signatures, exhibits) → token and compute savings.
- **If multi-evidences are far apart**, we need diversity-aware retrieval (MMR / reranking). If they're nearby, vanilla top-k with cosine similarity is enough.
- **If `evidence_text` is almost the whole page**, full-page chunking works. If it's a small fragment, we need finer-grained chunks.

In [ ]:
# === Block 1: Distribution of evidence_page_num ===
# Collect ALL page numbers from ALL evidence items (we don't aggregate per QA).
# Defensive: some evidence items may have page_num = None — skip those.
all_pages = []
for r in train:
    for item in r["evidence"]:
        page = item.get("evidence_page_num")
        if page is not None:
            all_pages.append(page)

print("=" * 75)
print(f"EVIDENCE_PAGE_NUM (n={len(all_pages)} total items across 150 QAs)")
print("=" * 75)
print(f"min={min(all_pages):3d}  p25={int(stats.quantiles(all_pages, n=4)[0]):3d}  "
      f"median={int(stats.median(all_pages)):3d}  p75={int(stats.quantiles(all_pages, n=4)[2]):3d}  "
      f"p95={int(stats.quantiles(all_pages, n=20)[18]):3d}  max={max(all_pages):3d}  "
      f"mean={int(stats.mean(all_pages)):3d}")
print()

# Manual histogram by page ranges that map to typical 10-K sections.
# The ranges are NOT uniform — they're chosen to separate natural 10-K items
# (Item 1 ~pp.0-25, MD&A ~pp.26-50, Financial Statements ~pp.51-75, etc.).
ranges = [(0, 25), (26, 50), (51, 75), (76, 100), (101, 150), (151, 250), (251, 500)]
print("Distribution by page range:")
for lo, hi in ranges:
    count = sum(1 for p in all_pages if lo <= p <= hi)
    pct = count / len(all_pages) * 100
    bar = "█" * int(pct / 2)
    print(f"  pp.{lo:3d}-{hi:3d}  {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 2: Multi-evidence locality ===
# For QAs with 2+ evidences, compute the "distance" between pages: max(pp) - min(pp).
#   distance = 0 → all evidences on the same page
#   distance = 5 → evidences within a 5-page window
# Tells whether multi-evidences are "clustered" (same section) or "scattered".
multi_qa_distances = []
for r in train:
    if len(r["evidence"]) >= 2:
        pages = [
            item["evidence_page_num"]
            for item in r["evidence"]
            if item.get("evidence_page_num") is not None
        ]
        if len(pages) >= 2:
            distance = max(pages) - min(pages)
            multi_qa_distances.append(distance)

print("=" * 75)
print(f"MULTI-EVIDENCE LOCALITY (n={len(multi_qa_distances)} QAs with 2+ evidences)")
print("=" * 75)
print("Distance between evidences (max page - min page):")
print(f"  min={min(multi_qa_distances):3d}  p25={int(stats.quantiles(multi_qa_distances, n=4)[0]):3d}  "
      f"median={int(stats.median(multi_qa_distances)):3d}  p75={int(stats.quantiles(multi_qa_distances, n=4)[2]):3d}  "
      f"max={max(multi_qa_distances):3d}  mean={int(stats.mean(multi_qa_distances)):3d}")
print()

dist_buckets = [
    (0, 0, "same page"),
    (1, 5, "near (1-5 pp)"),
    (6, 25, "far (6-25 pp)"),
    (26, 100, "very far (26-100 pp)"),
    (101, 1000, "extreme (>100 pp)"),
]
print("Distance distribution:")
for lo, hi, label in dist_buckets:
    count = sum(1 for d in multi_qa_distances if lo <= d <= hi)
    pct = count / len(multi_qa_distances) * 100
    bar = "█" * int(pct)  # 1:1 scale (each block = 1%) — the tail is small, this reads better
    print(f"  {label:25s} {count:3d}  {bar} {pct:.1f}%")
print()

# === Block 3: Ratio evidence_text vs evidence_text_full_page ===
# Each evidence item carries 2 strings:
#   evidence_text           → the RELEVANT passage (human-curated)
#   evidence_text_full_page → the full page where the passage lives
# The ratio tells what fraction of the page is the "real" evidence:
#   ratio close to 1.0 → the evidence IS almost the whole page (page = table)
#   ratio close to 0.0 → the evidence is a small fragment of a dense page
ratios = []
for r in train:
    for item in r["evidence"]:
        et = len(item.get("evidence_text", ""))
        full = len(item.get("evidence_text_full_page", ""))
        if full > 0:
            ratios.append(et / full)

print("=" * 75)
print(f"RATIO evidence_text / evidence_text_full_page (n={len(ratios)} items)")
print("=" * 75)
print(f"min={min(ratios):.2f}  p25={stats.quantiles(ratios, n=4)[0]:.2f}  "
      f"median={stats.median(ratios):.2f}  p75={stats.quantiles(ratios, n=4)[2]:.2f}  "
      f"max={max(ratios):.2f}  mean={stats.mean(ratios):.2f}")
print()

ratio_buckets = [
    (0.0, 0.25, "small fragment (<25%)"),
    (0.25, 0.5, "medium fragment (25-50%)"),
    (0.5, 0.9, "majority (50-90%)"),
    (0.9, 1.01, "almost full page (>90%)"),
]
print("Ratio distribution:")
for lo, hi, label in ratio_buckets:
    count = sum(1 for r in ratios if lo <= r < hi)
    pct = count / len(ratios) * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:30s} {count:3d}  {bar} {pct:.1f}%")

#### Findings

**1. ~85% of evidences live between pages 0-75** of the 10-K:

| Range | % | Maps to typical 10-K section |
|---|---:|---|
| pp. 0-25 | 27.5% | Item 1 (Business) + Item 1A (Risk Factors) |
| pp. 26-50 | 19.0% | Item 7 (MD&A) opening |
| **pp. 51-75** ⭐ | **38.1%** | **Item 8 (Financial Statements)** — the heart of the 10-K |
| pp. 76-100 | 7.9% | Notes to Financial Statements |
| pp. 101+ | 7.5% | Appendices and exhibits |

`mean ≈ median` (both = 51) → fairly symmetric distribution, no tails distorting the center.

**2. Multi-evidence locality: 91% are within ≤5 pages of each other**:

When a QA has 2-3 evidences, they almost always come from the **same section** of the 10-K (e.g., both from the Cash Flow Statement, or both from the Balance Sheet).

**Technical implication**: we do NOT need **MMR** (Maximal Marginal Relevance) or diverse reranking for the typical case — the relevant chunks are clustered, and top-k with cosine similarity captures them well. Only the ~9% of outliers (multi-evidence cross-section) would require something more sophisticated.

**3. Bimodal distribution of evidence/page ratio**:

- **46% of evidences cover almost the whole page** (>90%) — typical when the page is a complete financial table (Income Statement, Balance Sheet).
- **29% are small fragments** (<25%) — typical when the page has multiple topics and the evidence is a specific paragraph.

**Critical trade-off the chunking-strategy comparison will measure**:

| Strategy | How it does |
|---|---|
| Chunk = full page | ✅ Perfect for 46% (tables) · ❌ Dilutes small fragments |
| Chunk = 512 tokens | ✅ Captures small fragments · ❌ May split large tables |
| **Semantic chunking** | ✅ Respects semantic boundaries |
| **Late chunking** | ✅ Embeds the whole doc first, decides chunks afterward |

**This is why the project uses 4 strategies** instead of committing to one — each has strengths in different parts of the distribution.

> 📝 These measurements will be documented in `docs/chunking_decisions.md` (sub-block 6) when we justify which strategy wins in each ratio quartile.

---

## 🧭 Sub-block 2 wrap-up — what we learned about the dataset

After exploring the 5 sub-sections, we can tell the story of FinanceBench as a single coherent piece:

### 1. The dataset is a serious benchmark, not a toy example

The 150 questions are not "financial trivia". They're real queries a junior analyst would ask at an investment bank or fund:

- **32 distinct companies** across **9 economic sectors** (everything except Energy and Real Estate)
- **10 years of coverage** (2015-2024), though biased toward 2022-2023
- Balanced mix of three question types: **templated** (calculate ratios), **broad-domain** (strategy, risks), and **novel** (no fixed pattern)

> **Translation**: if a RAG system works here, there's a good chance it'll work on any real-world corporate financial domain.

### 2. The system doesn't just have to "find" — it has to understand

The `question_reasoning` field reveals three difficulty levels:

- **~21% direct lookup**: *"what was the revenue?"* — the system just needs to find the number.
- **~29% require calculation**: *"how much did it grow YoY?"* — the system finds two numbers and subtracts them.
- **~17% multi-step reasoning**: *"did CapEx grow faster than revenue?"* — multiple extractions, ratios, comparison.

> **Translation**: our project focuses only on the first half of the job (finding the right chunk). The calculation and reasoning are the downstream LLM's responsibility, which lives in Eslabón 2 of the roadmap.

### 3. There's a clear geographic pattern within the 10-K

The `evidence` items aren't randomly scattered — they live mostly in a specific zone of the document:

- **85% in the first 75 pages** of the 10-K
- Strong concentration in **pages 51-75**: the heart of the document (financial statements: Income, Balance Sheet, Cash Flow)
- When there are multiple evidences per question (~23% of dataset), **91% are within ≤5 pages of each other** — same section

> **Translation**: there's no need to read the entire 10-K to answer. Most answers live in a predictable zone. This justifies future optimizations (e.g., indexing only pages 0-150) and allows simple retrieval (vanilla top-k) instead of complex diversity algorithms.

### 4. The dataset is honest about its own complexity

The `evidence_text_full_page` field reveals a critical trade-off:

- **46% of evidences are a full page** (typically a financial table)
- **29% are small fragments** within dense pages
- The rest fall in the middle

> **Translation**: there's NO single chunking strategy that wins all cases. If we chunk to full page, we fail with fragments. If we chunk small, we split tables. **That's why the project compares 4 strategies** — each one optimized for a different dataset profile.

---

### What technical decisions this data justifies

| Technical decision | Justified by Sub-block 2 |
|---|---|
| Report **4 metrics** (Recall@k, MRR, NDCG, MAP) | Multi-evidence exists (~23%), MRR misleads, MAP captures everything |
| Report multiple `k` values (k=1, 3, 5, 10) | Evidence may not fit in 1 chunk (long-tail length distribution) |
| Compare **4 chunking strategies** | Bimodal evidence/page ratio distribution (no one-size-fits-all) |
| Keep the dataset **mixed** (don't filter to 10-K only) | Filtering would make us incomparable with academic literature |
| **Don't** invest in MMR / diverse retrieval for the baseline | Multi-evidences are nearby in 91% of cases |

This closes **Sub-block 2** of the FinanceBench Loader block.